# Log
* trim each column
* replace the value in ID column from _ to - to follow id in product table
* rename each column

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
px_cat_g1v2 = spark.table("workspace.default.erp_px_cat_g1v2")

px_cat_g1v2.display()

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products

In [0]:
%sql
SELECT DISTINCT cat 
FROM workspace.default.erp_px_cat_g1v2

In [0]:
%sql
 SELECT DISTINCT SUBCAT  FROM workspace.default.erp_px_cat_g1v2 

In [0]:
#trim
for field in px_cat_g1v2.schema.fields:
    if isinstance(field.dataType, StringType):
        px_cat_g1v2 = px_cat_g1v2.withColumn(field.name, trim(col(field.name)))

In [0]:
px_cat_g1v2 = px_cat_g1v2.withColumn("ID", F.regexp_replace("ID", "_", "-"))

In [0]:
rename_map_px_cat_g1v2 = {
        "ID": "category_id",
        "CAT": "category",
        "SUBCAT": "sub_category",
        "MAINTENANCE": "maintenance_flag"
    }
for old_name,new_name in rename_map_px_cat_g1v2.items():
    px_cat_g1v2 = px_cat_g1v2.withColumnRenamed(old_name,new_name)




In [0]:
px_cat_g1v2.display()

In [0]:
px_cat_g1v2.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_px_cat_g1v2")